# **SOLUTION**

In [11]:
class Solution(object):
    def assignEdgeWeights(self, edges, queries):
        """
        :type edges: List[List[int]]
        :type queries: List[List[int]]
        :rtype: List[int]
        """
        MOD = 10**9 + 7

        return self.solverLCABinLifting(edges, queries, MOD)

    def solverNaiveLCA(self, edges, queries, MOD):
        nE = len(edges)
        nV = nE + 1

        adjList = [[] for _ in range(nV + 1)]

        for x, y in edges:
            adjList[x].append(y)
            adjList[y].append(x)

        parent = [0] * (nV + 1)
        depth = [0] * (nV + 1)

        stack = [(1, 0, 0)]  # node, parent, depth

        while stack:
            node, p, d = stack.pop()

            parent[node] = p
            depth[node] = d

            for neighbor in adjList[node]:
                if neighbor != p:
                    stack.append((neighbor, node, d + 1))

        def getLCA(u, v):
            while depth[u] > depth[v]:
                u = parent[u]

            while depth[v] > depth[u]:
                v = parent[v]

            while u != v:
                u = parent[u]
                v = parent[v]

            return u

        results = []

        for u, v in queries:
            if u == v:
                results.append(0)
                continue

            lca = getLCA(u, v)

            dist = (
                depth[u]
                + depth[v]
                - 2 * depth[lca]
            )

            results.append(pow(2, dist - 1, MOD))

        return results

    def solverLCABinLifting(self, edges, queries, MOD):
        nE = len(edges)
        nV = nE + 1

        adjList = [[] for _ in range(nV + 1)]

        for x, y in edges:
            adjList[x].append(y)
            adjList[y].append(x)


        LOG = nV.bit_length()

        depth = [0] * (nV + 1)

        up = [[0] * (nV + 1) for _ in range(LOG)]

        stack = [(1, 0)]

        while stack:
            node, parent = stack.pop()

            up[0][node] = parent

            for neighbor in adjList[node]:
                if neighbor == parent:
                    continue

                depth[neighbor] = depth[node] + 1
                stack.append((neighbor, node))

        for k in range(1, LOG):
            for node in range(1, nV + 1):
                middleParent = up[k - 1][node]
                up[k][node] = up[k - 1][middleParent]

        def getLCA(u, v):
            if depth[u] < depth[v]:
                u, v = v, u

            depthDifference = depth[u] - depth[v]

            for k in range(LOG):
                if depthDifference & (1 << k):
                    u = up[k][u]

            if u == v:
                return u

            for k in range(LOG - 1, -1, -1):
                if up[k][u] != up[k][v]:
                    u = up[k][u]
                    v = up[k][v]

            return up[0][u]

        powersOfTwo = [1] * (nV + 1)

        for i in range(1, nV + 1):
            powersOfTwo[i] = (
                powersOfTwo[i - 1] * 2
            ) % MOD

        results = []

        for u, v in queries:
            if u == v:
                results.append(0)
                continue

            lca = getLCA(u, v)

            dist = (
                depth[u]
                + depth[v]
                - 2 * depth[lca]
            )

            results.append(powersOfTwo[dist - 1])

        return results

# **DRAFT**

# **TESTCASES**

In [12]:
solution = Solution()

# def extraCondition(result, expected):
#     pass


def run_test_case(case_number, input_edges, input_queries, expected):
    result = solution.assignEdgeWeights(input_edges, input_queries)
    status = "PASSED" if result == expected else "FAILED"
    # status = "PASSED" if extraCondition(result, expected) else "FAILED"
    print(f"Case {case_number} - (edges, queries): {(input_edges, input_queries)}, Output: {result}, Expected: {expected}, Status: {status}")

test_cases = [
    (1, [[1,2]], [[1,1],[1,2]], [0,1]),
    (2, [[1,2],[1,3],[3,4],[3,5]], [[1,4],[3,4],[2,5]], [2,1,4]),
]

for case_number, input_edges, input_queries, expected in test_cases:
    run_test_case(case_number, input_edges, input_queries, expected)


Case 1 - (edges, queries): ([[1, 2]], [[1, 1], [1, 2]]), Output: [0, 1], Expected: [0, 1], Status: PASSED
Case 2 - (edges, queries): ([[1, 2], [1, 3], [3, 4], [3, 5]], [[1, 4], [3, 4], [2, 5]]), Output: [2, 1, 4], Expected: [2, 1, 4], Status: PASSED
